# CAEM Experiment Notebook
**Session 21 — Experiment Phase**

Works on:
- **Local GPU** (RTX 3060, 12 GB VRAM — recommended for initial runs)
- **Google Colab Pro+** (A100, 40 GB VRAM — for full 5000-sample runs)
- **University cluster** (SLURM + any CUDA GPU)

Hardware is detected automatically via `scripts/hardware.py`.

### Experiment sequence
1. Setup — install deps, clone repo, detect hardware
2. Download datasets (HotpotQA, TruthfulQA, FEVER, StrategyQA)
3. Cycle 0 — baseline evaluation
4. Calibration — temperature scaling + signal weights
5. Cycles 1–3 — self-improvement + evaluation
6. Purity validation — three theory protocols
7. Ablation studies — 6 baselines + 3 ablation variants
8. Results summary — mechanism evidence table (Chapter 5)

## Cell 1: Environment setup

In [ ]:
# ── Detect environment ─────────────────────────────────────────────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL = not IN_COLAB

print(f'Environment: {"Google Colab" if IN_COLAB else "Local / Cluster"}')

# ── Clone repo (Colab only) ────────────────────────────────────────────────
if IN_COLAB:
    import subprocess
    repo_url = 'https://github.com/aksaN000/caem-thesis.git'
    if not os.path.exists('/content/caem-thesis'):
        subprocess.run(['git', 'clone', repo_url, '/content/caem-thesis'], check=True)
    os.chdir('/content/caem-thesis')
    print('Repo cloned and cwd set.')
else:
    # Already in the repo root on local machine
    repo_root = os.path.abspath('.')
    if 'caem-thesis' not in repo_root and os.path.exists('caem'):
        pass  # already in repo root
    elif os.path.exists('../caem'):
        os.chdir('..')
    print(f'Working directory: {os.getcwd()}')
    sys.path.insert(0, os.getcwd())

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
# Run once. Skip if already installed.

INSTALL_DEPS = True  # Set to False to skip on re-runs

if INSTALL_DEPS:
    import subprocess

    # Core ML deps
    packages = [
        'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
        'transformers>=4.36',
        'sentence-transformers',
        'faiss-gpu',   # Use 'faiss-cpu' on CPU-only machines
        'datasets',    # HuggingFace datasets
        'scipy',       # For temperature scaling (L-BFGS)
        'scikit-learn',# For signal weight calibration (logistic regression)
        'numpy',
        'tqdm',
    ]

    for pkg in packages:
        print(f'Installing: {pkg[:50]} ...')
        result = subprocess.run(
            f'pip install -q {pkg} --break-system-packages',
            shell=True, capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f'  WARNING: {result.stderr[:200]}')
        else:
            print(f'  OK')

    print('\nAll packages installed.')
else:
    print('Skipping installation (INSTALL_DEPS=False).')

In [ ]:
# ── Hardware detection ─────────────────────────────────────────────────────
from scripts.hardware import print_hardware_summary, apply_memory_flags

profile = print_hardware_summary()
apply_memory_flags(profile)

# Override batch size if VRAM is tight
# profile.recommended_batch_size = 2  # Uncomment for < 10 GB VRAM

## Cell 2: Dataset download

In [ ]:
# ── Download and cache all 4 benchmarks ───────────────────────────────────
# This runs once and caches to ~/.cache/huggingface/datasets/
# Takes ~5-10 min depending on connection speed.

from eval.benchmarks import load_hotpotqa, load_truthfulqa, load_fever, load_strategyqa

N_FULL = 5000   # Full experiment: 5000 per benchmark
N_TEST = 100    # Quick smoke test

# Change to N_TEST for a quick sanity check first
N = N_FULL

print(f'Loading benchmarks with N={N} per benchmark ...')
samples = {
    'hotpotqa':   load_hotpotqa(n=N),
    'truthfulqa': load_truthfulqa(n=N),
    'fever':      load_fever(n=N),
    'strategyqa': load_strategyqa(n=N),
}

for bm, s in samples.items():
    print(f'  {bm:<14}: {len(s)} samples loaded')

print('\nDatasets ready.')

## Cell 3: Build pipeline

In [ ]:
# ── Load models and build CAEMPipeline ────────────────────────────────────
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration, AutoModelForSequenceClassification
from caem.config import CAEMConfig
from caem.memory.encoder import QueryEncoder
from caem.pipeline import CAEMPipeline

device = profile.device
config = CAEMConfig()

# Override batch size from hardware profile
config.batch_size = profile.recommended_batch_size

# ── Flan-T5-Large ──────────────────────────────────────────────────────────
print('Loading Flan-T5-Large ...')
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-large')
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-large')

# Apply precision from hardware profile
if profile.use_bf16:
    model = model.to(torch.bfloat16)
    print('  Using bfloat16 (A100-class GPU)')
elif profile.use_fp16:
    model = model.half()
    print(f'  Using fp16 (VRAM: {profile.vram_gb:.1f} GB)')
else:
    print('  Using fp32 (CPU or debug mode)')

model = model.to(device).eval()
params_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  Flan-T5-Large: {params_m:.0f} M parameters')

# ── SBERT Encoder ──────────────────────────────────────────────────────────
print('Loading SBERT encoder (all-mpnet-base-v2) ...')
encoder = QueryEncoder(config=config)
print('  Done.')

# ── NLI model (RoBERTa-Large-MNLI) ─────────────────────────────────────────
print('Loading RoBERTa-Large-MNLI ...')
nli_tokenizer = AutoTokenizer.from_pretrained('roberta-large-mnli')
nli_model = AutoModelForSequenceClassification.from_pretrained('roberta-large-mnli')
if profile.use_fp16:
    nli_model = nli_model.half()
nli_model = nli_model.to(device).eval()
print('  Done.')

# ── Passage store (optional — requires pre-built Wikipedia FAISS index) ────
# Run scripts/build_passage_index.py first to build this.
passage_store = None
PASSAGE_INDEX_PATH = 'outputs/passage_index'
if __import__('os').path.exists(PASSAGE_INDEX_PATH):
    from caem.retrieval.rag import PassageStore
    passage_store = PassageStore.load(PASSAGE_INDEX_PATH)
    print(f'Passage store loaded ({len(passage_store)} passages).')
else:
    print(f'WARNING: No passage index at {PASSAGE_INDEX_PATH}.')
    print('Tier 3 RAG will run WITHOUT retrieved passages (lower accuracy).')
    print('Build with: python -m scripts.build_passage_index')

# ── Build pipeline ─────────────────────────────────────────────────────────
pipeline = CAEMPipeline(
    model=model,
    tokenizer=tokenizer,
    encoder=encoder,
    nli_model=nli_model,
    nli_tokenizer=nli_tokenizer,
    passage_store=passage_store,
    config=config,
    current_cycle=0,
)
print('\nCAEMPipeline ready.')

# Quick sanity check
test_result = pipeline.answer('Who wrote Hamlet?')
print(f'  Sanity check → "{test_result.answer}" (Tier {test_result.tier})')

## Cell 4: Dataset splits and eval harness

In [ ]:
# ── Non-overlapping splits (§5.3 of thesis plan) ───────────────────────────
# purity_samples  : first 500  → Theory 1/2/3 validation
# calib_samples   : next  500  → temperature scaling + signal weight calibration
# eval_samples    : remainder  → benchmark evaluation (reported in Chapter 5)

PURITY_SIZE = 500
CALIB_SIZE  = 500

purity_samples = {bm: s[:PURITY_SIZE] for bm, s in samples.items()}
calib_samples  = {bm: s[PURITY_SIZE:PURITY_SIZE+CALIB_SIZE] for bm, s in samples.items()}
eval_samples   = {bm: s[PURITY_SIZE+CALIB_SIZE:] for bm, s in samples.items()}

for bm in samples:
    print(f'{bm:<14}: purity={len(purity_samples[bm])}  '
          f'calib={len(calib_samples[bm])}  eval={len(eval_samples[bm])}')

# ── EvalHarness ────────────────────────────────────────────────────────────
import os
from eval.harness import EvalHarness
from caem.training.self_improvement import SelfImprovementLoop

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/eval', exist_ok=True)

harness = EvalHarness(pipeline, output_dir=f'{OUTPUT_DIR}/eval', log_every=200)

# Self-improvement loop
sil = SelfImprovementLoop(
    model=pipeline.model,
    tokenizer=pipeline.tokenizer,
    config=config,
    output_dir=OUTPUT_DIR,
)

print('Harness and SelfImprovementLoop ready.')

## Cell 5: General-domain data (anti-forgetting mix)

In [ ]:
from scripts.run_experiment import load_general_data
general_data = load_general_data(n=1000)
print(f'General-domain mix: {len(general_data)} QA pairs loaded.')

## Cell 6: Cycle 0 — Baseline evaluation

In [ ]:
import time, json

print('=' * 60)
print('CYCLE 0 — Baseline (zero episodic memory)')
print('=' * 60)

t0 = time.time()
cycle0_results = harness.run_all(eval_samples, cycle=0)
elapsed = (time.time() - t0) / 60
print(f'Cycle 0 done in {elapsed:.1f} min.')

# Display summary
print('\nCycle 0 results:')
for bm, res in sorted(cycle0_results.items()):
    print(f'  {bm:<14}: EM={res["em"]:.4f}  F1={res["f1"]:.4f}  '
          f'T1={res["tier1_frac"]*100:.1f}%  T3={res["tier3_frac"]*100:.1f}%')

all_cycle_results = [cycle0_results]

## Cell 7: Calibration (after Cycle 0)

In [ ]:
from scripts.run_calibration import calibrate_pipeline
from pathlib import Path

print('Running temperature scaling + signal weight calibration ...')
calib_result = calibrate_pipeline(
    pipeline=pipeline,
    calib_samples=calib_samples,
    config=config,
    output_dir=Path(f'{OUTPUT_DIR}/calibration'),
)
print('Calibration complete.')
print(f'  Temperature T     : {calib_result.get("temperature_scalar", "n/a")}')
print(f'  ECE before → after: {calib_result.get("ece_before", 0):.6f} → {calib_result.get("ece_after", 0):.6f}')
print(f'  Signal weights    : {calib_result.get("signal_weights", {})}')

## Cell 8: Cycles 1–3 (self-improvement loop)

In [ ]:
import json
from scripts.run_experiment import retroactive_reverification

NUM_CYCLES = 3

for cycle_num in range(1, NUM_CYCLES + 1):
    print('─' * 60)
    print(f'CYCLE {cycle_num} — Fine-tuning + evaluation')
    print('─' * 60)
    t0 = time.time()

    # Step 1: Fine-tune on verified memory episodes
    print(f'  Step 1: SelfImprovementLoop.run_cycle({cycle_num}) ...')
    cycle_result = sil.run_cycle(
        cycle_num=cycle_num,
        memory_store=pipeline.memory_store,
        general_data=general_data,
    )
    status = 'ABORTED (weights restored)' if cycle_result.aborted else 'OK'
    print(f'    Fine-tune: {status} | episodes={cycle_result.n_episodes_used} | '
          f'forgetting={cycle_result.forgetting_score:.4f} | loss={cycle_result.final_train_loss:.4f}')

    # Step 2: Update cycle counter
    pipeline.current_cycle = cycle_num

    # Step 3: Retroactive re-verification
    print(f'  Step 2: Retroactive re-verification ...')
    rv_stats = retroactive_reverification(pipeline, cycle_num, config)
    print(f'    Retroverify: total={rv_stats["total"]} | updated={rv_stats["updated"]} | pruned={rv_stats["pruned"]}')

    # Save retroverify stats
    with open(f'{OUTPUT_DIR}/retroverify_cycle{cycle_num}.json', 'w') as f:
        json.dump({'cycle': cycle_num, 'retroverify': rv_stats,
                   'fine_tune': {'n_episodes_used': cycle_result.n_episodes_used,
                                 'forgetting_score': cycle_result.forgetting_score,
                                 'aborted': cycle_result.aborted}}, f, indent=2)

    # Step 4: Evaluate all benchmarks
    print(f'  Step 3: Evaluating all benchmarks (cycle={cycle_num}) ...')
    cycle_results = harness.run_all(eval_samples, cycle=cycle_num)
    all_cycle_results.append(cycle_results)

    elapsed = (time.time() - t0) / 60
    print(f'Cycle {cycle_num} done in {elapsed:.1f} min.')

    for bm, res in sorted(cycle_results.items()):
        print(f'  {bm:<14}: EM={res["em"]:.4f}  F1={res["f1"]:.4f}  '
              f'T1={res["tier1_frac"]*100:.1f}%  T3={res["tier3_frac"]*100:.1f}%')

# Save full results
with open(f'{OUTPUT_DIR}/all_cycle_results.json', 'w') as f:
    json.dump(all_cycle_results, f, indent=2)
print('\nAll cycle results saved → outputs/all_cycle_results.json')

## Cell 9: Mechanism evidence table (Chapter 5, Table 1)

In [ ]:
from scripts.run_experiment import print_mechanism_table, save_summary_csv
from pathlib import Path

print_mechanism_table(all_cycle_results)
save_summary_csv(all_cycle_results, Path(OUTPUT_DIR))
print(f'Summary CSV saved → {OUTPUT_DIR}/experiment_summary.csv')

## Cell 10: Theory validation (Chapter 5, Section 5.5)

In [ ]:
# NOTE: Theory validation requires one pipeline per cycle.
# Since we only have one pipeline object (weights updated in-place),
# full multi-cycle theory validation needs checkpoints loaded separately.
# 
# Quick Theory 1 check using current (Cycle 3) pipeline:

from scripts.run_purity_validation import run_purity_validation_protocol
from pathlib import Path

# Use current pipeline for a single-cycle purity validation
theory_results = run_purity_validation_protocol(
    pipelines_by_cycle={3: pipeline},   # cycle 3 only; extend with loaded checkpoints for full T2/T3
    purity_samples=purity_samples,
    output_dir=Path(f'{OUTPUT_DIR}/purity_validation'),
)

# For full multi-cycle validation run:
# python -m scripts.run_purity_validation --checkpoints_dir outputs --num_cycles 3

## Cell 11: Ablation studies

In [ ]:
# Full ablation study — run standalone for clean comparison:
# python -m scripts.run_ablation --cycle3_checkpoint outputs/cycle_3 --n_questions 500
#
# Quick zero-shot baseline inline:
import torch
from scripts.run_ablation import ZeroShotBaseline, eval_baseline
from pathlib import Path

print('Running zero-shot baseline (A0) for comparison...')
zs = ZeroShotBaseline(pipeline.model, pipeline.tokenizer, device)

# Use a small subset for speed
n_ablation = 200
ablation_samples = {bm: s[:n_ablation] for bm, s in eval_samples.items()}
zs_results = eval_baseline('zero_shot', zs, ablation_samples, Path(f'{OUTPUT_DIR}/ablation'))

print('\nZero-shot vs CAEM Cycle 3 (first {} samples):'.format(n_ablation))
for bm in sorted(ablation_samples):
    zs_em = zs_results.get(bm, {}).get('em', 0)
    caem_em = all_cycle_results[-1].get(bm, {}).get('em', 0)  # cycle 3
    improvement = (caem_em - zs_em) / max(zs_em, 0.001) * 100
    print(f'  {bm:<14}: ZS={zs_em:.4f}  CAEM={caem_em:.4f}  Δ={improvement:+.1f}%')

## Cell 12: MMLU Retention check

In [ ]:
from scripts.run_ablation import eval_mmlu_retention

print('Measuring MMLU retention (target ≥ 0.93) ...')
mmlu_score = eval_mmlu_retention(pipeline, n=200)
status = '✓ OK' if mmlu_score >= 0.93 else '✗ BELOW TARGET — review EWC λ'
print(f'MMLU retention: {mmlu_score:.4f}  {status}')

## Cell 13: Save all outputs

Download from Colab: Files panel → right-click `outputs/` → Download as zip.

In [ ]:
import os, json

# List all output files
print('Output files:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in sorted(files):
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, OUTPUT_DIR)
        print(f'  outputs/{rel:<55} {size_kb:>8.1f} KB')

# Colab: zip and download
if IN_COLAB:
    import subprocess
    subprocess.run(['zip', '-r', '/content/caem_outputs.zip', OUTPUT_DIR], check=True)
    from google.colab import files
    files.download('/content/caem_outputs.zip')
    print('Downloading outputs as ZIP...')

---
## Quick reference

### Standalone scripts (from repo root)
```bash
# Full experiment (Cycle 0→3, all benchmarks)
python -m scripts.run_experiment --n_questions 5000 --output_dir outputs

# Smoke test (CPU, synthetic data, no downloads)
python -m scripts.run_experiment --smoke_test

# Calibration only (after Cycle 0)
python -m scripts.run_calibration --output_dir outputs/calibration

# Ablation studies
python -m scripts.run_ablation --cycle3_checkpoint outputs/cycle_3 --n_questions 500

# Theory validation (requires all cycle checkpoints)
python -m scripts.run_purity_validation --checkpoints_dir outputs --num_cycles 3
```

### Hardware flags
```bash
# Force CPU (for debugging)
CAEM_DEVICE=cpu python -m scripts.run_experiment --smoke_test

# Disable NLI model (saves ~1.5 GB VRAM)
python -m scripts.run_experiment --no_nli

# Disable RAG (saves time if no passage index)
python -m scripts.run_experiment --no_rag
```

### Expected outputs
```
outputs/
  eval/
    hotpotqa_cycle0.json     # EvalResult + SampleResults
    hotpotqa_cycle1.json
    ...                      # 4 benchmarks × 4 cycles = 16 files
  calibration/
    calibrated_config.json   # T + signal weights (actual, report in Ch5)
  cycle_1/
    model.pt                 # Fine-tuned Flan-T5-Large weights
    cycle_metadata.json
  cycle_2/ ... cycle_3/
  purity_validation/
    theory_validation.json   # Theory 1, 2, 3 tables
  ablation/
    baseline_zero_shot.json
    ablation_summary.json
  all_cycle_results.json     # Master results file
  experiment_summary.csv     # Chapter 5 mechanism evidence table
```